# Project - Airline AI Assistant

We'll now bring together what we've learned to make an AI Customer Support assistant for an Airline

In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [2]:
# Initialization

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
deepseek = os.getenv("DEEPSEEK_API_KEY")

if deepseek:
    print(f"Deepseek API Key exists and begins {deepseek[:8]}")
elif openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "deepseek-flash"
openai = OpenAI(base_url="https://api.deepseek.com", api_key=deepseek)

# As an alternative, if you'd like to use Ollama instead of OpenAI
# Check that Ollama is running for you locally (see week1/day2 exercise) then uncomment these next 2 lines
# MODEL = "llama3.2"
# openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')


Deepseek API Key exists and begins sk-f730a


In [3]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [4]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Tools

Tools are an incredibly powerful feature provided by the frontier LLMs.

With tools, you can write a function, and have the LLM call that function as part of its response.

Sounds almost spooky.. we're giving it the power to run code on our machine?

Well, kinda.

In [6]:
# Let's start by making a useful function

ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
    print(f"Tool called for city {destination_city}")
    price = ticket_prices.get(destination_city.lower(), "Unknown ticket price")
    return f"The price of a ticket to {destination_city} is {price}"


In [8]:
get_ticket_price("London")

Tool called for city London


'The price of a ticket to London is $799'

In [7]:
# There's a particular dictionary structure that's required to describe our function:

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

In [9]:
# And this is included in a list of tools:

tools = [{"type": "function", "function": price_function}]

In [10]:
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Get the price of a return ticket to the destination city.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['destination_city'],
    'additionalProperties': False}}}]

## Getting OpenAI to use our Tool

There's some fiddly stuff to allow OpenAI "to call our tool"

What we actually do is give the LLM the opportunity to inform us that it wants us to run the tool.

Here's how the new chat function looks:

In [41]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    if response.choices[0].finish_reason == 'tool_calls':
        msg = response.choices[0].message
        tool_response = handle_tool_call(response.choices[0].message)
        messages.append(msg)
        messages.extend(tool_response) # if we use append there would be a nested list. better use extend
        print(f"messages: {messages}")
        response = openai.chat.completions.create(model=MODEL, messages=messages)

    print(f"Response: {response}")
    return response.choices[0].message.content

In [39]:
def handle_tool_call(msg):
    # we want to get the function argument from the message which contain the desination, and we also need the function name which is from function.name
    response = []
    for tool_call in msg.tool_calls:
        print(f"Tool call: {tool_call}")
        arg = json.loads(tool_call.function.arguments)
        city = arg.get("destination_city")
        price = get_ticket_price(city)
        tool_id = tool_call.id
        response.append({
            "role": "tool",
            "content": price,
            "tool_call_id": tool_id
        })
    return response
    

In [ ]:
# We have to write that function handle_tool_call:

def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    if tool_call.function.name == "get_ticket_price":
        arguments = json.loads(tool_call.function.arguments)
        city = arguments.get('destination_city')
        price_details = get_ticket_price(city)
        response = {
            "role": "tool",
            "content": price_details,
            "tool_call_id": tool_call.id
        }
    return response

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7876
* To create a public link, set `share=True` in `launch()`.


Tool call: ChatCompletionMessageFunctionToolCall(id='call_00_50qRfnjHXkOD99J2d6qq6950', function=Function(arguments='{"destination_city": "Paris"}', name='get_ticket_price'), type='function', index=0)
Tool called for city Paris
Tool call: ChatCompletionMessageFunctionToolCall(id='call_01_DVZMnjdZrrMZeQ5EVI0p3007', function=Function(arguments='{"destination_city": "London"}', name='get_ticket_price'), type='function', index=1)
Tool called for city London
messages: [{'role': 'system', 'content': "\nYou are a helpful assistant for an Airline called FlightAI.\nGive short, courteous answers, no more than 1 sentence.\nAlways be accurate. If you don't know the answer, say so.\n"}, {'role': 'user', 'content': 'i need ticket to paris and london'}, ChatCompletionMessage(content="I'll look up ticket prices for both destinations.", refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_00_50qRfnjHXkOD99J2d6qq6950

## Let's make a couple of improvements

Handling multiple tool calls in 1 response

Handling multiple tool calls 1 after another

In [ ]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content

In [ ]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

In [ ]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [47]:
# https://docs.python.org/3/library/sqlite3.html
# https://neon.com/postgresql/tutorial/real-data-type
# REAL data type allows you to store single-precision floating-point numbers in the database.
import sqlite3

conn = sqlite3.connect('flight_price.db')
cur = conn.cursor()
cur.execute('CREATE TABLE IF NOT EXISTS flight_prices (city TEXT PRIMARY KEY, price REAL)')
conn.commit()

In [54]:
def get_ticket_price(city):
    conn = sqlite3.connect('flight_price.db')
    cur = conn.cursor()
    cur.execute("SELECT price FROM flight_prices WHERE city=?", (city.lower(),))
    result = cur.fetchone()
    print(f"result {result}")
    return f"Ticket price to {city} is {result[0]}" if result else f"No price data found for {city}"

In [ ]:
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

In [ ]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [55]:
get_ticket_price("London")

result None


'No price data found for London'

In [ ]:
def set_ticket_price(city, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()

In [59]:
def set_ticket_price(city, price):
    conn = sqlite3.connect("flight_price.db")
    cur = conn.cursor()
    cur.execute('INSERT INTO flight_prices (city, price) VALUES (?, ?) ON CONFLICT (city) DO UPDATE SET price = ?', (city.lower(), price, price))
    conn.commit()

In [60]:
ticket_prices = {"london":799, "paris": 899, "tokyo": 1420, "sydney": 2999, "jakarta": 10}
for city, price in ticket_prices.items():
    set_ticket_price(city, price)

In [61]:
get_ticket_price("Tokyo")

result (1420,)


'Ticket price to Tokyo is 1420'

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7877
* To create a public link, set `share=True` in `launch()`.


Tool call: ChatCompletionMessageFunctionToolCall(id='call_00_osf5hyjsKk5nmZpTcjrA7763', function=Function(arguments='{"destination_city": "Jakarta"}', name='get_ticket_price'), type='function', index=0)
result (10,)
Tool call: ChatCompletionMessageFunctionToolCall(id='call_01_bZKI4b10wCatJRwa9Rdq5292', function=Function(arguments='{"destination_city": "Tokyo"}', name='get_ticket_price'), type='function', index=1)
result (1420,)
messages: [{'role': 'system', 'content': "\nYou are a helpful assistant for an Airline called FlightAI.\nGive short, courteous answers, no more than 1 sentence.\nAlways be accurate. If you don't know the answer, say so.\n"}, {'role': 'user', 'content': 'get ticket to jakarta and tokyo'}, ChatCompletionMessage(content="I'll check both prices for you.", refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_00_osf5hyjsKk5nmZpTcjrA7763', function=Function(arguments='{"destination_

## Exercise

Add a tool to set the price of a ticket!

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business Applications</h2>
            <span style="color:#181;">Hopefully this hardly needs to be stated! You now have the ability to give actions to your LLMs. This Airline Assistant can now do more than answer questions - it could interact with booking APIs to make bookings!</span>
        </td>
    </tr>
</table>